# IRIS — Neural IDS for AI Agent Pipelines

**One-click launcher** for the IRIS interactive dashboard.

This notebook clones the project from GitHub, installs dependencies, downloads pre-trained checkpoints, and launches the Gradio web app with a **public URL** you can open in any browser.

### What you need
- A Google Colab runtime with a **T4 GPU** (free tier works)
- ~60 seconds for model loading after setup

### Instructions
1. Select `Runtime > Change runtime type > T4 GPU`
2. Select `Runtime > Run all`
3. **First time only:** the runtime will restart automatically (numpy downgrade). When it does, click `Runtime > Run all` again — the second run is fast.

*Nathan Cheung () | York University | CSSD 2221 | Winter 2026*

In [ ]:
# === Mount Google Drive (for caching) and clone repository ===
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ncheung13579/iris.git /content/iris 2>/dev/null || echo "Already cloned."
%cd /content/iris

# Install dependencies. numpy is downgraded from Colab's 2.x to 1.x
# (required by transformer-lens), so a runtime restart is needed once.
import importlib, numpy
needs_restart = int(numpy.__version__[0]) >= 2
!pip install -r requirements.txt -q
if needs_restart:
    print("\n** numpy was downgraded — restarting runtime. Re-run this cell (it will be fast). **")
    import os; os.kill(os.getpid(), 9)

In [ ]:
# === Download pre-trained checkpoints (with Drive cache) ===
# First run: downloads from GitHub Release → caches to Drive.
# Subsequent runs: restores from Drive instantly, even after runtime reset.
# If Drive is full, caching is skipped — the download still works.
import os, shutil, urllib.request

RELEASE_URL = "https://github.com/ncheung13579/iris/releases/download/v1.0.0"
DRIVE_CACHE = "/content/drive/MyDrive/iris_checkpoints"

files = {
    "checkpoints/sae_d10240_lambda1e-04.pt": "sae_d10240_lambda1e-04.pt",
    "checkpoints/feature_matrix.npy": "feature_matrix.npy",
    "checkpoints/sensitivity_scores.npy": "sensitivity_scores.npy",
    "data/processed/iris_dataset_balanced.json": "iris_dataset_balanced.json",
}

os.makedirs(DRIVE_CACHE, exist_ok=True)

for local_path, remote_name in files.items():
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    drive_path = os.path.join(DRIVE_CACHE, remote_name)

    if os.path.exists(local_path):
        # Already in the working directory (re-run within same runtime)
        print(f"  [OK] {local_path} (already exists)")
    elif os.path.exists(drive_path):
        # Restore from Drive cache (runtime recycled but Drive persists)
        shutil.copy2(drive_path, local_path)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        print(f"  [OK] {local_path} (restored from Drive, {size_mb:.1f} MB)")
    else:
        # First time: download from GitHub Release
        url = f"{RELEASE_URL}/{remote_name}"
        print(f"  Downloading {remote_name}...", end=" ", flush=True)
        urllib.request.urlretrieve(url, local_path)
        size_mb = os.path.getsize(local_path) / (1024 * 1024)
        # Cache to Drive (best-effort — skip if Drive is full)
        try:
            shutil.copy2(local_path, drive_path)
            print(f"OK ({size_mb:.1f} MB, cached to Drive)")
        except OSError:
            print(f"OK ({size_mb:.1f} MB, Drive cache skipped — low storage)")

print("\nAll checkpoints ready.")

In [ ]:
# === Launch the IRIS Detection Dashboard ===
# This loads GPT-2, the SAE, and all detectors (~60 seconds),
# then opens a Gradio web app with a public URL.
from src.app import launch
launch(project_root='.', share=True)